# Tuần 07 — Thực hành k-Means: 3 bài

Demo `03_Demo-kMeans.ipynb` gom cụm bằng tay trên dữ liệu giả 2 chiều.

File này có **3 bài**. Mỗi bài là một bộ dữ liệu thật trên Kaggle. Cả 3 bài đều làm. Dùng `sklearn.cluster.KMeans`.

Clustering không chia train/test và không có nhãn để học. `StandardScaler` fit trên toàn bộ bảng feature. Không đưa cột ID hay tên vào khoảng cách.

Tải 3 file vào `data/`:

| Bài | Bộ dữ liệu | Link | File |
|---|---|---|---|
| 1 | Mall Customer Segmentation | https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python | `data/Mall_Customers.csv` |
| 2 | Unsupervised Learning on Country Data | https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data | `data/Country-data.csv` |
| 3 | Wholesale Customers | https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set | `data/Wholesale customers data.csv` |

Ba bộ này cũng được dùng ở `09_TH-DBSCAN.ipynb`, để so sánh hai cách gom cụm trên cùng dữ liệu.


In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)


## Phần chung

Elbow: với mỗi `K` từ 1 đến `k_max`, fit `KMeans` (`n_init=10`, `random_state=3000`), lưu `inertia_`, rồi vẽ. Chỗ khuỷu tay là `K` nên thử.

Sau khi có nhãn, bảng trung bình theo cụm giúp đọc ý nghĩa từng nhóm.


In [2]:
def ve_elbow(X_scaled, k_max=10):
    ks = list(range(1, k_max + 1))
    inertias = []
    for k in ks:
        # TODO: fit KMeans với n_clusters=k, n_init=10, random_state=3000
        # lưu model.inertia_ vào inertias
       model = KMeans(n_clusters=k, n_init=10, random_state=3000)
       model.fit(X_scaled)
       inertias.append(model.inertia_)
    plt.figure(figsize=(6, 4))
    plt.plot(ks, inertias, marker='o')
    plt.xlabel('K')
    plt.ylabel('inertia')
    plt.grid(True)
    plt.show()
    return ks, inertias


def bang_trung_binh(df_goc, labels, cot_so):
    tam = df_goc[cot_so].copy()
    tam['cum'] = labels
    return tam.groupby('cum').mean().round(2)


---

# Bài 1 — Khách hàng trung tâm thương mại

Nguồn: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python

200 khách. Cột: `CustomerID`, `Gender`, `Age`, `Annual Income (k$)`, `Spending Score (1-100)`.

Bài này chỉ dùng **thu nhập** và **điểm chi tiêu**. Bỏ `CustomerID`. Không đưa `Gender` vào khoảng cách. Có thể để `Age` sang một lần chạy riêng nếu còn thời gian.

Chuẩn hóa hai cột, vẽ elbow, chọn `K`, tô màu scatter theo cụm. Dấu `x` là tâm cụm tính trên **số gốc** (trung bình thu nhập và điểm chi tiêu của các điểm trong cụm), để đọc được đơn vị thật.


In [10]:
mall = pd.read_csv('data/marketing_campaign.csv')
print(mall.head())
print(mall.shape)

cot = ['Annual Income (k$)', 'Spending Score (1-100)']
X = mall[cot].to_numpy(dtype=float)
X_scaled = StandardScaler().fit_transform(X)

ve_elbow(X_scaled)


  ID\tYear_Birth\tEducation\tMarital_Status\tIncome\tKidhome\tTeenhome\tDt_Customer\tRecency\tMntWines\tMntFruits\tMntMeatProducts\tMntFishProducts\tMntSweetProducts\tMntGoldProds\tNumDealsPurchases\tNumWebPurchases\tNumCatalogPurchases\tNumStorePurchases\tNumWebVisitsMonth\tAcceptedCmp3\tAcceptedCmp4\tAcceptedCmp5\tAcceptedCmp1\tAcceptedCmp2\tComplain\tZ_CostContact\tZ_Revenue\tResponse
0  5524\t1957\tGraduation\tSingle\t58138\t0\t0\t0...                                                                                                                                                                                                                                                                                                                                                  
1  2174\t1954\tGraduation\tSingle\t46344\t1\t1\t0...                                                                                                                                                                      

KeyError: "None of [Index(['Annual Income (k$)', 'Spending Score (1-100)'], dtype='str')] are in the [columns]"

In [3]:
# TODO: chọn K từ elbow
K = 5
model = KMeans(n_clusters=K, n_init=10, random_state=3000)
labels = model.fit_predict(X_scaled)

plt.figure(figsize=(6, 6))
for cum in np.unique(labels):
    pts = X[labels == cum]
    plt.scatter(pts[:, 0], pts[:, 1], label=f'cụm {cum}')
    plt.scatter(pts[:, 0].mean(), pts[:, 1].mean(), c='black', marker='x', s=120)
plt.xlabel(cot[0])
plt.ylabel(cot[1])
plt.legend()
plt.grid(True)
plt.show()

print(bang_trung_binh(mall, labels, cot))
print('số điểm mỗi cụm:', np.unique(labels, return_counts=True))


NameError: name 'X_scaled' is not defined

**Nộp kèm bài 1**

1. `K` bạn chọn là bao nhiêu? Elbow gãy ở đâu?
2. Mô tả từng cụm bằng thu nhập và điểm chi tiêu (cao / thấp). Cụm nào đáng ưu tiên cho khuyến mãi?
3. Nếu thêm cột `Age` vào `X` rồi chuẩn hóa lại, bạn có đổi `K` không?


In [ ]:
# Bài 1
# 1.Chọn K = 5. Elbow thường bắt đầu giảm chậm hơn rõ rệt quanh K = 5, vì vậy chọn 5 cụm là hợp lý.
# 2.Các cụm có thể được phân biệt dựa trên thu nhập và điểm chi tiêu.
# - Thu nhập thấp, điểm chi tiêu thấp: nhóm ít giá trị.
# - Thu nhập thấp, điểm chi tiêu cao: có khả năng khai thác bằng các chương trình khuyến mãi phù hợp.
# - Thu nhập trung bình, điểm chi tiêu trung bình: nhóm khách hàng phổ thông.
# - Thu nhập cao, điểm chi tiêu thấp: có khả năng chi trả nhưng chưa mua sắm nhiều.
# - Thu nhập cao, điểm chi tiêu cao: nhóm khách hàng giá trị cao.
# 3.Nếu thêm Age thì cần chạy lại elbow vì việc thêm một đặc trưng có thể làm thay đổi cấu trúc khoảng cách giữa các điểm dữ liệu. Tuy nhiên không nên tự động đổi K; cần quan sát biểu đồ elbow mới rồi quyết định.


---

# Bài 2 — Phân nhóm quốc gia

Nguồn: https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data

167 nước, file `Country-data.csv`. Cột `country` là tên, không đưa vào `X`.

Các cột số: `child_mort`, `exports`, `health`, `imports`, `income`, `inflation`, `life_expec`, `total_fer`, `gdpp`.

`income` và `gdpp` lớn hơn `child_mort` rất nhiều. Bắt buộc `StandardScaler`. Chọn `K` bằng elbow (thường thử quanh 3). In trung bình `child_mort`, `income`, `life_expec`, `gdpp` theo cụm, và liệt kê khoảng 8 tên nước mỗi cụm.


In [11]:
country = pd.read_csv('data/Country-data.csv')
print(country.head())
print(country.shape)

cot_so = [c for c in country.columns if c != 'country']
# TODO: X từ cot_so, chuẩn hóa
X = country[cot_so].to_numpy(dtype=float)
X_scaled = StandardScaler().fit_transform(X)

ve_elbow(X_scaled)


FileNotFoundError: [Errno 2] No such file or directory: 'data/Country-data.csv'

In [ ]:
K = 3
labels = KMeans(n_clusters=K, n_init=10, random_state=3000).fit_predict(X_scaled)

print(bang_trung_binh(country, labels, ['child_mort', 'income', 'life_expec', 'gdpp']))

for cum in np.unique(labels):
    ten = country.loc[labels == cum, 'country'].head(8).tolist()
    print(cum, ten)


**Nộp kèm bài 2**

1. `K` bạn chọn? Cụm nào có `child_mort` cao và `gdpp` thấp?
2. Kể tên 3 nước thuộc cụm đó và 3 nước thuộc cụm có `gdpp` cao.
3. Vì sao không chuẩn hóa thì `income` và `gdpp` sẽ át các cột còn lại?


In [ ]:
# Bài 2
# 1.Chọn K = 3. Cụm có child_mort cao và gdpp thấp là nhóm các quốc gia có mức phát triển kinh tế thấp. Nhóm này thường có thu nhập thấp, tuổi thọ thấp và tỷ lệ tử vong trẻ em cao.
# 2.
# 3.Vì các cột income và gdpp có giá trị lớn hơn rất nhiều so với nhiều biến khác. KMeans sử dụng khoảng cách để xác định các điểm gần nhau, nên những biến có độ lớn lớn sẽ đóng góp nhiều hơn vào khoảng cách.StandardScaler đưa các biến về cùng thang đo, giúp mỗi đặc trưng có ảnh hưởng hợp lý hơn đến quá trình phân cụm.


---

# Bài 3 — Khách sỉ

Nguồn: https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set

440 khách hàng bán sỉ. File `Wholesale customers data.csv`.

Cột: `Channel`, `Region`, `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`.

`Channel` (1 = Horeca, 2 = Retail) và `Region` là mã nhóm, không phải số tiền. **Không** đưa hai cột này vào `X`. Gom cụm trên 6 cột chi tiêu, sau chuẩn hóa. Rồi dùng `pd.crosstab` để xem mỗi cụm nằm ở kênh nào.


In [ ]:
ws = pd.read_csv('data/Wholesale customers data.csv')
print(ws.head())
print(ws.shape)

cot_chi = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
# TODO: chuẩn hóa 6 cột chi tiêu
X = ws[cot_chi].to_numpy(dtype=float)
X_scaled = StandardScaler().fit_transform(X)

ve_elbow(X_scaled)


In [ ]:
K = 4
labels = KMeans(n_clusters=K, n_init=10, random_state=3000).fit_predict(X_scaled)

print(bang_trung_binh(ws, labels, cot_chi))
print(pd.crosstab(labels, ws['Channel'], rownames=['cum'], colnames=['Channel']))


**Nộp kèm bài 3**

1. `K` bạn chọn? Cụm nào chi mạnh cho `Grocery` và `Detergents_Paper`?
2. Cụm đó lệch về `Channel` nào trên bảng crosstab?
3. So với bài 1: vì sao bài 3 không vẽ một scatter là đủ để đọc cụm?


In [ ]:
# Bài 3
# 1.Chọn K = 4.Dựa vào bảng trung bình theo cụm, tìm cụm có giá trị trung bình của Grocery và Detergents_Paper cao nhất. Đây là cụm có mức chi tiêu mạnh cho hai nhóm hàng này.
# 2.
# 3.Bài 1 chỉ sử dụng 2 đặc trưng là Annual Income và Spending Score nên có thể biểu diễn trực tiếp trên mặt phẳng 2 chiều bằng scatter plot.Bài 3 sử dụng 6 đặc trưng chi tiêu gồm Fresh, Milk, Grocery, Frozen, Detergents_Paper và Delicassen. Vì dữ liệu có nhiều chiều nên một scatter 2 chiều không thể thể hiện đầy đủ cấu trúc của các cụm. Do đó cần sử dụng bảng trung bình theo cụm và crosstab để phân tích kết quả.
